# Feature Engineering & Preprocessing Pipeline
Building a robust, production-ready machine learning pipeline for the Customer Churn Prediction System.

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Add utils to path if needed (we are running from root or notebooks dir)
import sys
import os
sys.path.append(os.path.abspath('..'))
from utils.preprocessing import load_dataset, engineer_features, split_dataset, build_preprocessor, transform_dataset, save_pipeline

## 2. Load Dataset & Feature Inspection

In [2]:
# Load dataset
df = load_dataset('../dataset/clean_customer_churn.csv')
df.head()

,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,...,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,86,5372,Moved
3,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,1,89,5340,Competitor had better devices


In [3]:
# Display shape
print(f"Dataset Shape: {df.shape}")

# Display columns
print("\nColumns:")
print(df.columns.tolist())

# Display datatypes
print("\nDatatypes:")
print(df.dtypes)

# Display missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Display unique values
print("\nUnique Values per column:")
print(df.nunique())

Dataset Shape: (7032, 32)

Columns:
['Count', 'Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude', 'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Label', 'Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']

Datatypes:
Count                  int64
Country                  str
State                    str
City                     str
Zip Code               int64
Lat Long                 str
Latitude             float64
Longitude            float64
Gender                   str
Senior Citizen           str
Partner                  str
Dependents               str
Tenure Months          int64
Phone Service            str
Multiple Lines           str
Internet Service         str
Online Secu

## 3. Target Variable Analysis

In [4]:
# Check churn distribution and class imbalance
churn_dist = df['Churn Label'].value_counts()
churn_percentages = df['Churn Label'].value_counts(normalize=True) * 100

print("Churn Distribution:")
print(churn_dist)
print("\nClass Imbalance (Percentages):")
print(churn_percentages)

Churn Distribution:
Churn Label
No     5163
Yes    1869
Name: count, dtype: int64

Class Imbalance (Percentages):
Churn Label
No     73.421502
Yes    26.578498
Name: proportion, dtype: float64


## 4. Feature Engineering
Creating meaningful engineered features: TotalCharges numeric conversion, tenure groups, MonthlyCharges categories, TotalCharges categories, and Contract length features.

In [5]:
# Apply feature engineering
# Engineered Features Explained:
# - Tenure Group: Categorizes continuous tenure months into lifecycle stages.
# - Monthly Charge Group / Total Charge Group: Bins financial data to capture price sensitivity thresholds.
# - Contract Length (Months): Converts categorical contract types into a meaningful numeric duration.
df = engineer_features(df)
print("Features engineered successfully.")
df.head()

Features engineered successfully.


,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,...,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason,Tenure Group,Monthly Charge Group,Total Charge Group,Contract Length (Months)
0,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,...,108.15,Yes,1,86,3239,Competitor made better offer,0-12,Medium,Low,1
1,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,...,151.65,Yes,1,67,2701,Moved,0-12,High,Low,1
2,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,...,820.50,Yes,1,86,5372,Moved,0-12,Premium,Low,1
3,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,...,3046.05,Yes,1,84,5003,Moved,25-36,Premium,High,1
4,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,...,5036.30,Yes,1,89,5340,Competitor had better devices,49-60,Premium,Very High,1


## 5 & 6. Setup Encoding and Scaling
Automatically detect Categorical and Numerical features to apply OneHotEncoder and StandardScaler.

In [6]:
# We will target 'Churn Label' as our main target (or 'Churn Value')
target_col = 'Churn Label'

# Drop secondary target columns to prevent data leakage
cols_to_drop = ['Churn Value', 'Churn Score', 'CLTV', 'Churn Reason']
for col in cols_to_drop:
    if col in df.columns:
        df = df.drop(columns=[col])

# Exclude target from features
X_temp = df.drop(columns=[target_col])

# Automatically detect numerical and categorical features
numerical_features = X_temp.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_temp.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numerical Features ({len(numerical_features)}): {numerical_features}")
print(f"Categorical Features ({len(categorical_features)}): {categorical_features}")

Numerical Features (8): ['Count', 'Zip Code', 'Latitude', 'Longitude', 'Tenure Months', 'Monthly Charges', 'Total Charges', 'Contract Length (Months)']
Categorical Features (23): ['Country', 'State', 'City', 'Lat Long', 'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', 'Tenure Group', 'Monthly Charge Group', 'Total Charge Group']


## 7. Train Test Split
Splitting the data 80/20 with stratification and random state 42.

In [7]:
# Split dataset
X_train, X_test, y_train, y_test = split_dataset(df, target_col=target_col, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (5625, 31)
X_test shape: (1407, 31)


## 8. Build Preprocessing Pipeline
Using ColumnTransformer and Pipeline for reusability.

In [8]:
# Build the preprocessor
preprocessor = build_preprocessor(numerical_features, categorical_features)
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

## 10. Transform Datasets

In [9]:
# Transform data
X_train_transformed, X_test_transformed, preprocessor_fitted = transform_dataset(preprocessor, X_train, X_test)

print("Data transformed successfully.")
print(f"Transformed X_train shape: {X_train_transformed.shape}")

Data transformed successfully.
Transformed X_train shape: (5625, 2843)


## 9. Save Preprocessor
Save `models/preprocessor.pkl` using Joblib.

In [10]:
# Save pipeline
save_pipeline(preprocessor_fitted, '../models/preprocessor.pkl')

Pipeline saved to ../models/preprocessor.pkl


## 11. Save Transformed Data
Saving X_train, X_test, y_train, y_test to CSV files in `dataset/`.

In [11]:
# Save to CSV
X_train_transformed.to_csv('../dataset/X_train.csv', index=False)
X_test_transformed.to_csv('../dataset/X_test.csv', index=False)
y_train.to_csv('../dataset/y_train.csv', index=False)
y_test.to_csv('../dataset/y_test.csv', index=False)

print("✅ Transformed datasets saved successfully to 'dataset/' directory.")

✅ Transformed datasets saved successfully to 'dataset/' directory.
